# 02 · Preparación COCO Estados Unidos

Detector activo: **RT-DETRv2-S** con **OpenVINO Runtime nativo en Python**, checkpoint `merve/license-plates-rtdetrv2` con licencia Apache-2.0. Los datos y métricas nuevos se mantienen separados de los resultados históricos YOLO.


In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1" "openvino>=2025.4,<2027"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="usa"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Perfil:",COUNTRY,"| RT-DETRv2-S:",MODEL_ID)


## Formato de entrenamiento

El entrenamiento USA de RT-DETR usa cajas en COCO Detection. La fuente Roboflow configurada es el proyecto [European + USA License plates](https://universe.roboflow.com/plateblur/european-usa-license-plates) (`plateblur/european-usa-license-plates`), versión 1. Roboflow lo publica como Instance Segmentation, así que se descarga en COCO Segmentation y se usan las cajas `bbox`; las máscaras se ignoran. Las etiquetas `Plate`, `license-plate` y `Plate B` se consolidan en una sola clase `license_plate`, como espera el detector.

La versión 1 reserva 106 imágenes para test según su ficha de Roboflow; úsalo para una revisión inicial. Para las métricas finales, prepara un test independiente mayor y evita separar imágenes de una misma secuencia o vehículo entre particiones.

El convertidor del proyecto también puede transformar etiquetas heredadas a COCO, sin Ultralytics ni inferencia YOLO.

In [ ]:
from src.train import (
    RF_FORMAT,
    RF_PROJECT,
    RF_TITLE,
    RF_VERSION,
    RF_WORKSPACE,
    _annotation_file,
    convert_yolo_to_coco,
    download_roboflow_dataset,
)

DATA_DIR = None
USE_ROBOFLOW = True

print(
    f"Fuente Roboflow: {RF_TITLE} | "
    f"{RF_WORKSPACE}/{RF_PROJECT} | versión {RF_VERSION} | {RF_FORMAT}"
)

if USE_ROBOFLOW:
    # Configura ROBOFLOW_API_KEY en Secrets de Colab.
    DATA_DIR = str(download_roboflow_dataset("/content/datasets"))

if DATA_DIR:
    for split in ("train", "val", "test"):
        try:
            print(split, _annotation_file(DATA_DIR, split))
        except FileNotFoundError as exc:
            print(exc)


In [ ]:
# Conversión opcional desde anotaciones YOLO previas, si no hay fuente COCO.
DATA_YAML=None  # ejemplo: /content/dataset/data.yaml
if DATA_YAML:
    DATA_DIR=str(convert_yolo_to_coco(DATA_YAML,"/content/dataset-coco"))
    print("Dataset convertido:",DATA_DIR)
